# 动手练习：疾病模块、网络邻近度与药物重定位

来自《生物医学网络科学》（Biomedical Network Science）。 参考答案版。

## 任务 1：疾病基因是否聚成模块：最大连通分支与度匹配随机基线

合成“相互作用组”：1000 个节点的 BA 无标度网络。疾病模块：从一个中等度数的种子出发、每步加入与模块连边最多的邻居，长成 25 个节点；为模拟“数据不完整”，只知道其中 20 个疾病基因（`disease`），另外 5 个藏在 `hidden` 里，任务 4 会用到。数据只用于演示方法，不代表任何真实蛋白或疾病。

计算疾病基因诱导子图的最大连通分支（LCC）大小 `obs`，再与 300 组**度匹配**的随机基因集比较：随机集合中每个基因都从与原基因同一个度箱（`node2bin[基因]`）里抽取，整体不重复。得到 $z=(S-\langle S^{\mathrm{rand}}\rangle)/\sigma(S^{\mathrm{rand}})$。

要得到的变量：`rand_sizes`（300 个随机 LCC 大小）、`z_lcc`。

期望看到：疾病基因的 LCC 远大于随机期望，$z$ 远大于 2（见 [疾病模块](network-medicine.disease-modules.html)）。

In [ ]:
# 准备（直接运行）
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt

def make_network(seed=14):
    G = nx.barabasi_albert_graph(1000, 2, seed=seed)
    rng = np.random.default_rng(seed)
    start = int(rng.choice([n for n in G if 4 <= G.degree(n) <= 8]))
    module = {start}
    while len(module) < 25:
        frontier = {v for u in module for v in G[u]} - module
        score = {v: sum(1 for u in G[v] if u in module) for v in frontier}
        best = max(score.values())
        module.add(int(rng.choice(sorted(v for v in frontier if score[v] == best))))
    known = sorted(rng.choice(sorted(module), size=20, replace=False).tolist())
    return G, known, sorted(module - set(known))

def degree_bins(G, min_bin=40):
    """按度从小到大合并相邻度值，直到每箱至少 min_bin 个节点。返回 节点 -> 所在箱的节点列表。"""
    by_deg = {}
    for n, d in G.degree():
        by_deg.setdefault(d, []).append(n)
    bins, cur = [], []
    for d in sorted(by_deg):
        cur += by_deg[d]
        if len(cur) >= min_bin:
            bins.append(cur); cur = []
    if cur:
        bins[-1] += cur
    return {n: b for b in bins for n in b}

def lcc_size(G, nodes):
    H = G.subgraph(nodes)
    return max(len(c) for c in nx.connected_components(H)) if len(H) else 0

G, disease, hidden = make_network()
node2bin = degree_bins(G)
obs = lcc_size(G, disease)

In [ ]:
rng = np.random.default_rng(0)
rand_sizes = []
for _ in range(300):
    rand_set = set()
    for s in disease:
        c = int(rng.choice(node2bin[s]))
        while c in rand_set:
            c = int(rng.choice(node2bin[s]))
        rand_set.add(c)
    rand_sizes.append(lcc_size(G, rand_set))
z_lcc = (obs - np.mean(rand_sizes)) / np.std(rand_sizes)
print("观测 LCC = %d，随机期望 = %.1f ± %.1f，z = %.1f" % (obs, np.mean(rand_sizes), np.std(rand_sizes), z_lcc))

In [ ]:
# 核对（不用修改）
assert len(rand_sizes) == 300, "rand_sizes 应有 300 个值"
assert max(rand_sizes) <= len(disease), "随机 LCC 不可能超过疾病基因数；检查是否每次都只抽了 len(disease) 个节点"
assert np.mean(rand_sizes) < obs / 2, "度匹配随机集的 LCC 应远小于观测值；检查是否从 node2bin[基因] 里抽样，而不是直接用了疾病基因"
assert z_lcc is not None and z_lcc > 2, "z 应远大于 2；检查公式 (obs - 均值) / 标准差"
print("✓ 任务 1 通过")

## 任务 2：药物–疾病网络邻近度：最近距离与 z 分数

$d_c(S,T)=\frac1{|T|}\sum_{t\in T}\min_{s\in S}d(s,t)$。`min_to_disease[v]` 已给出每个节点到最近疾病基因的距离。药物 A 的 3 个靶点与疾病基因直接相邻；药物 B 的 3 个靶点是全网度最大的枢纽。写出 `closest(targets)`，再用 500 组度匹配随机靶点计算两种药物的 $z$。

要得到的变量：函数 `closest`，以及 `dA`、`dB`（原始 $d_c$）和 `zA`、`zB`。

期望看到：药物 B 的原始距离也不大（枢纽离谁都近），但与同度数的随机靶点相比 $z$ 接近 0；药物 A 的 $z$ 明显为负（见 [网络邻近度](network-medicine.network-proximity.html)）。

In [ ]:
# 准备（直接运行）
N = G.number_of_nodes()
D = np.full((len(disease), N), np.inf)
for i, s in enumerate(disease):
    for v, d in nx.single_source_shortest_path_length(G, s).items():
        D[i, v] = d
min_to_disease = D.min(axis=0)
nbrs = sorted({v for s in disease for v in G[s]} - set(disease))
drug_A = [v for v in nbrs if 3 <= G.degree(v) <= 10][:3]
drug_B = [v for v, _ in sorted(G.degree(), key=lambda x: -x[1]) if v not in disease][:3]

def proximity_z(targets, n_rand=500, seed=1):
    rng = np.random.default_rng(seed)
    o = closest(targets)
    rand = [closest([int(rng.choice(node2bin[t])) for t in targets]) for _ in range(n_rand)]
    return o, (o - np.mean(rand)) / np.std(rand)

In [ ]:
def closest(targets):
    return float(np.mean(min_to_disease[list(targets)]))

dA, zA = proximity_z(drug_A)
dB, zB = proximity_z(drug_B)
print("药物 A：靶点度 %s，d_c = %.2f，z = %.2f" % ([G.degree(t) for t in drug_A], dA, zA))
print("药物 B：靶点度 %s，d_c = %.2f，z = %.2f" % ([G.degree(t) for t in drug_B], dB, zB))

In [ ]:
# 核对（不用修改）
assert closest([disease[0]]) == 0, "疾病基因本身到疾病模块的距离应为 0"
assert abs(closest(drug_A) - 1.0) < 1e-9, "药物 A 的靶点都与疾病基因直接相邻，d_c 应为 1；检查是否对 targets 取了平均"
assert zA is not None and zA < -1, "药物 A 的 z 应明显为负"
assert zB is not None and abs(zB) < 1.5, "药物 B（枢纽靶点）与同度数的随机靶点相比不更近，z 应接近 0；检查随机靶点是否来自 node2bin[t]"
print("✓ 任务 2 通过")

## 任务 3：两个疾病模块的网络分离度

$s_{AB}=\langle d_{AB}\rangle-\frac12(\langle d_{AA}\rangle+\langle d_{BB}\rangle)$。`mean_within(A, sp)` 是 A 中每个基因到 A 中其他基因最近距离的平均；`mean_between(A, B, sp)` 是 A、B 之间对称的最近距离平均（共有基因距离为 0）。疾病 B 以 A 的 12 个基因为起点生长，与 A 部分重叠；疾病 C 从离 A 最远的种子长出。

要得到的变量：函数 `separation(A, B)`，以及 `s_AB`、`s_AC`。

期望看到：$s_{AB}<0$（模块重叠），$s_{AC}>0$（模块分离），见 [网络分离度](network-medicine.network-separation.html)。

In [ ]:
# 准备（直接运行）
G3 = nx.barabasi_albert_graph(1000, 2, seed=14)
deg3 = dict(G3.degree())
rng3 = np.random.default_rng(3)

def grow(start, size, maxdeg=15):
    module = set(start) if isinstance(start, list) else {start}
    while len(module) < size:
        frontier = {v for u in module for v in G3[u] if deg3[v] <= maxdeg} - module
        score = {v: sum(1 for u in G3[v] if u in module) for v in frontier}
        best = max(score.values())
        module.add(int(rng3.choice(sorted(v for v in frontier if score[v] == best))))
    return sorted(module)

def mean_within(A, sp):
    return np.mean([min(sp[a][b] for b in A if b != a) for a in A])

def mean_between(A, B, sp):
    return np.mean([min(sp[a][b] for b in B) for a in A] + [min(sp[b][a] for a in A) for b in B])

seeds3 = [n for n in G3 if 4 <= deg3[n] <= 8]
dA_ = grow(seeds3[0], 20)
dB_ = grow(dA_[:12], 20)
spA = {s: nx.single_source_shortest_path_length(G3, s) for s in dA_}
to_A = {n: min(spA[a][n] for a in dA_) for n in G3}
dC_ = grow(max(seeds3, key=lambda n: to_A[n]), 20)
sp = {s: nx.single_source_shortest_path_length(G3, s) for s in set(dA_) | set(dB_) | set(dC_)}

In [ ]:
def separation(A, B):
    return mean_between(A, B, sp) - (mean_within(A, sp) + mean_within(B, sp)) / 2

s_AB = separation(dA_, dB_)
s_AC = separation(dA_, dC_)
print("A 与 B：共有基因 %d 个，s = %.2f；A 与 C：共有基因 %d 个，s = %.2f"
      % (len(set(dA_) & set(dB_)), s_AB, len(set(dA_) & set(dC_)), s_AC))

In [ ]:
# 核对（不用修改）
assert s_AB is not None and s_AB < 0, "A 与 B 部分重叠，s_AB 应为负；检查是否减去的是两个模块内距离的平均（除以 2）"
assert s_AC is not None and s_AC > 0, "A 与 C 相距很远，s_AC 应为正"
assert abs(separation(dA_, dA_) - (0 - mean_within(dA_, sp))) < 1e-9, "同一个模块与自身的分离度应为 -<d_AA>"
print("✓ 任务 3 通过")

## 任务 4：网络传播：用带重启的随机游走找回“缺失”的疾病基因

回到任务 1 的网络。以 20 个已知疾病基因为种子做带重启的随机游走（RWR）：$p_{t+1}=(1-r)Wp_t+rp_0$，$W=AD^{-1}$ 为列随机矩阵，$p_0$ 在种子上均匀，$r=0.3$，迭代到 $\|p_{t+1}-p_t\|_1<10^{-8}$。然后在**非种子**节点中，看 5 个隐藏的模块基因（`hidden`）能否排在前面：计算 AUROC（隐藏基因的分数高于其他非种子节点的概率）。再以“节点的度”作为打分，计算同样的 AUROC 作对照（见 [网络传播](network-medicine.network-propagation.html)）。

要得到的变量：`p_star`（长度 1000 的稳态分布）、`auc_rwr`、`auc_deg`。

期望看到：RWR 的 AUROC 接近 1，明显高于只按度排序。

> **提示：** `A = nx.to_numpy_array(G, nodelist=range(1000))`，`W = A / A.sum(axis=0)`。

In [ ]:
# 准备（直接运行）
def auroc(pos_scores, neg_scores):
    pos, neg = np.asarray(pos_scores)[:, None], np.asarray(neg_scores)[None, :]
    return float((pos > neg).mean() + 0.5 * (pos == neg).mean())

others = [v for v in range(1000) if v not in set(disease) | set(hidden)]
degree_arr = np.array([G.degree(v) for v in range(1000)], dtype=float)

In [ ]:
Aj = nx.to_numpy_array(G, nodelist=range(1000))
W = Aj / Aj.sum(axis=0)
p0 = np.zeros(1000); p0[disease] = 1 / len(disease)
p = p0.copy()
for _ in range(1000):
    p_new = 0.7 * W @ p + 0.3 * p0
    if np.abs(p_new - p).sum() < 1e-8:
        break
    p = p_new
p_star = p_new
auc_rwr = auroc(p_star[hidden], p_star[others])
auc_deg = auroc(degree_arr[hidden], degree_arr[others])
print("RWR AUROC = %.3f；按度排序 AUROC = %.3f；隐藏基因的度：%s" % (auc_rwr, auc_deg, degree_arr[hidden].astype(int).tolist()))

In [ ]:
# 核对（不用修改）
assert p_star is not None and len(p_star) == 1000, "p_star 应有 1000 个值"
assert abs(p_star.sum() - 1) < 1e-6, "RWR 的稳态分布应归一（和为 1）；检查 W 是否为列随机矩阵 A / A.sum(axis=0)"
assert p_star[disease].mean() > p_star[others].mean(), "种子上的概率应高于其他节点"
assert auc_rwr is not None and auc_rwr > 0.9, "RWR 应能把隐藏的模块基因排在前面（AUROC > 0.9）；检查重启项 r * p0"
assert auc_deg is not None and auc_deg < auc_rwr, "只按度排序的 AUROC 应低于 RWR"
print("✓ 任务 4 通过")

## 任务 5：开放小分析：真实相互作用组中的疾病模块与药物邻近度

数据：STRING v12.0 人类物理相互作用（分数 ≥ 700，`data/string_human_physical_v12_s700.csv`）与 Open Targets 的 11 种疾病遗传关联基因（`data/opentargets_disease_genes_ga.csv`）、17 种药物的作用机制靶点（`data/opentargets_drug_targets.csv`），来源见 [网络医学的真实数据案例](network-medicine.html)。setup 里已建好最大连通分支 `GR`（scipy 稀疏矩阵）、名字到下标的映射 `gid`、度分箱 `bins`/`bin_of`，以及 `lcc_size_r(S)` 与 `degree_matched_r(S, rng)`。

自己选一种疾病和遗传关联阈值（如 GA ≥ 0.7），回答：它的基因是否形成显著的模块？再选一种药物，算它的靶点到该疾病模块的最近距离 $d_c$。

要得到的变量：`disease_name`（字符串）、`lcc_obs`（整数）、`z_obs`（浮点数）、`dc_obs`（浮点数，可为 `np.inf` 表示不连通）。

思考：换成均匀随机基线，$z$ 变大还是变小？为什么？药物的靶点本身是不是该疾病的遗传关联基因？

In [ ]:
# 准备（直接运行）
import pandas as pd
from scipy import sparse
from scipy.sparse.csgraph import connected_components, shortest_path
ppi = pd.read_csv("data/string_human_physical_v12_s700.csv")
dg = pd.read_csv("data/opentargets_disease_genes_ga.csv")
drugs = pd.read_csv("data/opentargets_drug_targets.csv").fillna("")
nodes = pd.Index(sorted(set(ppi.protein1) | set(ppi.protein2)))
ix = pd.Series(np.arange(len(nodes)), index=nodes)
Ar = sparse.coo_matrix((np.ones(len(ppi)), (ix[ppi.protein1].values, ix[ppi.protein2].values)), shape=(len(nodes),) * 2)
Ar = ((Ar + Ar.T) > 0).astype(np.int8).tocsr()
_, comp = connected_components(Ar, directed=False)
keep = np.where(comp == np.bincount(comp).argmax())[0]
GR = Ar[keep][:, keep].tocsr()
gid = pd.Series(np.arange(len(keep)), index=nodes[keep])
degR = np.asarray(GR.sum(1)).ravel()
order = np.argsort(degR, kind="stable")
bins, cur = [], []
for pos, v in enumerate(order):
    cur.append(v)
    if len(cur) >= 100 and not (pos + 1 < len(order) and degR[order[pos + 1]] == degR[v]):
        bins.append(np.array(cur)); cur = []
if cur:
    bins[-1] = np.concatenate([bins[-1], cur])
bin_of = np.empty(len(degR), dtype=int)
for k_, arr in enumerate(bins):
    bin_of[arr] = k_

def lcc_size_r(S):
    S = np.asarray(S)
    if len(S) == 0:
        return 0
    _, c = connected_components(GR[S][:, S], directed=False)
    return int(np.bincount(c).max())

def degree_matched_r(S, rng):
    out = []
    for s in S:
        c = rng.choice(bins[bin_of[s]])
        while c in out:
            c = rng.choice(bins[bin_of[s]])
        out.append(c)
    return np.array(out)

print(sorted(dg.disease.unique()))
print(drugs[["drug", "targets"]].to_string(index=False))

In [ ]:
disease_name = "rheumatoid arthritis"
genes = dg[(dg.disease == disease_name) & (dg.ga_score >= 0.7)].gene
S = gid[[g for g in genes if g in gid.index]].values
lcc_obs = lcc_size_r(S)
rng5 = np.random.default_rng(5)
null5 = np.array([lcc_size_r(degree_matched_r(S, rng5)) for _ in range(200)])
z_obs = float((lcc_obs - null5.mean()) / null5.std())
T = [gid[t] for t in drugs.set_index("drug").loc["tofacitinib", "targets"].split(";") if t in gid.index]
dist_to_S = shortest_path(GR, directed=False, unweighted=True, indices=S).min(0)
dc_obs = float(dist_to_S[T].mean())
print("%s：%d 个基因，LCC = %d，度匹配 z = %.1f；托法替布靶点 d_c = %.2f" % (disease_name, len(S), lcc_obs, z_obs, dc_obs))

In [ ]:
# 核对（不用修改）
assert isinstance(disease_name, str) and disease_name in set(dg.disease), "disease_name 应是数据中的疾病名之一"
assert lcc_obs is not None and int(lcc_obs) >= 1, "lcc_obs 应是正整数"
assert z_obs is not None and np.isfinite(z_obs), "z_obs 应是有限的数"
assert dc_obs is not None and dc_obs >= 0, "d_c 是距离，不能为负"
print("✓ 任务 5 通过")